In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True

# ============================================================
# Исходные данные: вариант 35
# Переход в начало координат: (x1f, x2f) = (0, 0)
# ============================================================
J = 2500.0
M_max = 45.0

x10_deg = -40.0
x20_deg = 25.0

x10 = np.deg2rad(x10_deg)
x20 = np.deg2rad(x20_deg)
k = M_max / J

x1f = 0.0
x2f = 0.0

# Для данного варианта оптимальное управление состоит из двух участков.
mu1 = -1
mu2 = +1

# Первая фазовая траектория при mu = -1:
# x2^2 = -2*k*x1 + C1
C1 = x20**2 + 2 * k * x10

# Конечная фазовая траектория при mu = +1:
# x2^2 = 2*k*x1 + Cf
Cf = x2f**2 - 2 * k * x1f

# Точка переключения находится из пересечения:
# -2*k*x1 + C1 = 2*k*x1 + Cf
x1_sw = (C1 - Cf) / (4 * k)
x2_sw = -np.sqrt(Cf + 2 * k * x1_sw)

# Времена участков.
dt1 = (x20 - x2_sw) / k
dt2 = (x2f - x2_sw) / k
t_total = dt1 + dt2

print("Аналитический расчет для перехода в начало координат")
print("----------------------------------------------------")
print(f"k = {k:.6f} 1/с^2")
print(f"x10 = {x10:.6f} рад")
print(f"x20 = {x20:.6f} рад/с")
print(f"x1f = {x1f:.6f} рад")
print(f"x2f = {x2f:.6f} рад/с")
print(f"C1 = {C1:.9f}")
print(f"Cf = {Cf:.9f}")
print(f"x1_sw = {x1_sw:.9f} рад")
print(f"x2_sw = {x2_sw:.9f} рад/с")
print(f"dt1 = {dt1:.6f} с")
print(f"dt2 = {dt2:.6f} с")
print(f"t_total = {t_total:.6f} с")

# ============================================================
# Аналитические зависимости x1(t), x2(t)
# ============================================================
t1 = np.linspace(0, dt1, 600)
x1_1 = x10 + x20 * t1 - 0.5 * k * t1**2
x2_1 = x20 - k * t1

tau = np.linspace(0, dt2, 600)
x1_2 = x1_sw + x2_sw * tau + 0.5 * k * tau**2
x2_2 = x2_sw + k * tau

t2 = dt1 + tau

# ============================================================
# Фазовые диаграммы при постоянном управлении mu = +1 и mu = -1
# ============================================================
x1_min = min(-4.0, x10, x1_sw, x1f) - 0.5
x1_max = max(4.0, x10, x1_sw, x1f) + 0.5
x1_grid = np.linspace(x1_min, x1_max, 1600)
vertex_positions = np.linspace(x1_min + 0.4, x1_max - 0.4, 7)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for x1_vertex in vertex_positions:
    x2_squared = 2 * k * (x1_grid - x1_vertex)
    mask = x2_squared >= 0
    plt.plot(x1_grid[mask], np.sqrt(x2_squared[mask]), color="royalblue", alpha=0.8)
    plt.plot(x1_grid[mask], -np.sqrt(x2_squared[mask]), color="royalblue", alpha=0.8)

plt.title("Фазовые траектории при μ = +1")
plt.xlabel("x1, рад")
plt.ylabel("x2, рад/с")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)

plt.subplot(1, 2, 2)
for x1_vertex in vertex_positions:
    x2_squared = 2 * k * (x1_vertex - x1_grid)
    mask = x2_squared >= 0
    plt.plot(x1_grid[mask], np.sqrt(x2_squared[mask]), color="crimson", alpha=0.8)
    plt.plot(x1_grid[mask], -np.sqrt(x2_squared[mask]), color="crimson", alpha=0.8)

plt.title("Фазовые траектории при μ = -1")
plt.xlabel("x1, рад")
plt.ylabel("x2, рад/с")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.show()

# ============================================================
# Оптимальная фазовая траектория
# ============================================================
plt.figure(figsize=(8, 5))

plt.plot(x1_1, x2_1, label="1-й участок: μ = -1", linewidth=2.2)
plt.plot(x1_2, x2_2, label="2-й участок: μ = +1", linewidth=2.2)

# Конечная фазовая траектория, по которой система приходит в (0, 0).
x1_line = np.linspace(x1f, x1_sw, 600)
inside = x2f**2 + 2 * k * (x1_line - x1f)
x2_line = -np.sqrt(np.maximum(0.0, inside))
plt.plot(x1_line, x2_line, "--", color="black", label="линия переключения")

plt.scatter(x10, x20, label="начальная точка", zorder=5)
plt.scatter(x1_sw, x2_sw, label="точка переключения", zorder=5)
plt.scatter(x1f, x2f, label="конечная точка (0; 0)", zorder=5)

plt.title("Оптимальная фазовая траектория")
plt.xlabel("x1, рад")
plt.ylabel("x2, рад/с")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# Численное моделирование методом Рунге-Кутты 4-го порядка
# ============================================================
def system_right_part(state, mu):
    x1, x2 = state
    return np.array([x2, k * mu])


def rk4_step(state, mu, h):
    k1 = system_right_part(state, mu)
    k2 = system_right_part(state + 0.5 * h * k1, mu)
    k3 = system_right_part(state + 0.5 * h * k2, mu)
    k4 = system_right_part(state + h * k3, mu)
    return state + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6


def simulate_segment(state0, t0, duration, mu, h):
    t_values = [t0]
    x1_values = [state0[0]]
    x2_values = [state0[1]]
    mu_values = [mu]

    state = state0.copy()
    t = t0
    t_end = t0 + duration

    while t < t_end - 1e-12:
        h_step = min(h, t_end - t)
        state = rk4_step(state, mu, h_step)
        t += h_step

        t_values.append(t)
        x1_values.append(state[0])
        x2_values.append(state[1])
        mu_values.append(mu)

    return np.array(t_values), np.array(x1_values), np.array(x2_values), np.array(mu_values), state


def simulate(h=0.001):
    state0 = np.array([x10, x20], dtype=float)

    t_a, x1_a, x2_a, mu_a, state_sw_num = simulate_segment(state0, 0.0, dt1, mu1, h)
    t_b, x1_b, x2_b, mu_b, state_final_num = simulate_segment(state_sw_num, dt1, dt2, mu2, h)

    t_values = np.concatenate([t_a, t_b[1:]])
    x1_values = np.concatenate([x1_a, x1_b[1:]])
    x2_values = np.concatenate([x2_a, x2_b[1:]])
    mu_values = np.concatenate([mu_a, mu_b[1:]])

    return t_values, x1_values, x2_values, mu_values, state_sw_num, state_final_num


t_num, x1_num, x2_num, mu_num, state_sw_num, state_final_num = simulate(h=0.001)

print()
print("Численное моделирование")
print("-----------------------")
print(f"x1_sw_num = {state_sw_num[0]:.9f} рад")
print(f"x2_sw_num = {state_sw_num[1]:.9f} рад/с")
print(f"t_sw_num = {dt1:.6f} с")
print(f"x1_final_num = {state_final_num[0]:.9f} рад")
print(f"x2_final_num = {state_final_num[1]:.9f} рад/с")
print(f"t_final_num = {t_total:.6f} с")

# Переходные процессы x1(t), x2(t)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(t_num, x1_num, label="x1(t)")
plt.axvline(dt1, linestyle="--", label=f"переключение: {dt1:.2f} с")
plt.axhline(x1f, linestyle=":", label="x1f = 0 рад")
plt.title("Переходный процесс по x1")
plt.xlabel("t, с")
plt.ylabel("x1, рад")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t_num, x2_num, label="x2(t)")
plt.axvline(dt1, linestyle="--", label=f"переключение: {dt1:.2f} с")
plt.axhline(x2f, linestyle=":", label="x2f = 0 рад/с")
plt.title("Переходный процесс по x2")
plt.xlabel("t, с")
plt.ylabel("x2, рад/с")
plt.legend()

plt.tight_layout()
plt.show()

# Закон управления mu(t)
plt.figure(figsize=(8, 3))
plt.step(t_num, mu_num, where="post")
plt.axvline(dt1, linestyle="--", label="момент переключения")
plt.title("Оптимальное управление μ(t)")
plt.xlabel("t, с")
plt.ylabel("μ")
plt.yticks([-1, 0, 1])
plt.legend()
plt.tight_layout()
plt.show()

print()
print("Сравнение аналитических и численных значений")
print("--------------------------------------------")
print(f"x1_sw: аналитически {x1_sw:.9f}, численно {state_sw_num[0]:.9f}")
print(f"x2_sw: аналитически {x2_sw:.9f}, численно {state_sw_num[1]:.9f}")
print(f"x1_final: требуется {x1f:.9f}, численно {state_final_num[0]:.9f}")
print(f"x2_final: требуется {x2f:.9f}, численно {state_final_num[1]:.9f}")
print(f"dt1 = {dt1:.6f} с")
print(f"dt2 = {dt2:.6f} с")
print(f"t_total = {t_total:.6f} с")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True

# ============================================================
# Исходные данные: вариант 35
# Переход в точку: (x1f, x2f) = (0.5, 0.5)
# ============================================================
J = 2500.0
M_max = 45.0

x10_deg = -40.0
x20_deg = 25.0

x10 = np.deg2rad(x10_deg)
x20 = np.deg2rad(x20_deg)
k = M_max / J

x1f = 0.5
x2f = 0.5

# Для данного перехода оптимальное управление состоит из двух участков.
mu1 = -1
mu2 = +1

# Первая фазовая траектория при mu = -1:
# x2^2 = -2*k*x1 + C1
C1 = x20**2 + 2 * k * x10

# Конечная фазовая траектория при mu = +1:
# x2^2 = 2*k*x1 + Cf
Cf = x2f**2 - 2 * k * x1f

# Точка переключения находится из пересечения:
# -2*k*x1 + C1 = 2*k*x1 + Cf
x1_sw = (C1 - Cf) / (4 * k)
x2_sw = -np.sqrt(C1 - 2 * k * x1_sw)

# Вершина конечной фазовой траектории при mu = +1.
x1_vertex_final = -Cf / (2 * k)

# Времена участков.
dt1 = (x20 - x2_sw) / k
dt2 = (x2f - x2_sw) / k
t_total = dt1 + dt2

print("Аналитический расчет для перехода в точку (0.5; 0.5)")
print("------------------------------------------------------")
print(f"k = {k:.6f} 1/с^2")
print(f"x10 = {x10:.6f} рад")
print(f"x20 = {x20:.6f} рад/с")
print(f"x1f = {x1f:.6f} рад")
print(f"x2f = {x2f:.6f} рад/с")
print(f"C1 = {C1:.9f}")
print(f"Cf = {Cf:.9f}")
print(f"x1_sw = {x1_sw:.9f} рад")
print(f"x2_sw = {x2_sw:.9f} рад/с")
print(f"x1_vertex_final = {x1_vertex_final:.9f} рад")
print(f"dt1 = {dt1:.6f} с")
print(f"dt2 = {dt2:.6f} с")
print(f"t_total = {t_total:.6f} с")

# ============================================================
# Аналитические зависимости x1(t), x2(t)
# ============================================================
t1 = np.linspace(0, dt1, 900)
x1_1 = x10 + x20 * t1 - 0.5 * k * t1**2
x2_1 = x20 - k * t1

tau = np.linspace(0, dt2, 1200)
x1_2 = x1_sw + x2_sw * tau + 0.5 * k * tau**2
x2_2 = x2_sw + k * tau

t2 = dt1 + tau

# ============================================================
# Фазовые диаграммы при постоянном управлении mu = +1 и mu = -1
# ============================================================
x1_min = min(x10, x1_sw, x1f, x1_vertex_final) - 0.5
x1_max = max(x10, x1_sw, x1f, 1.0) + 0.5
x1_grid = np.linspace(x1_min, x1_max, 1800)
vertex_positions = np.linspace(x1_min + 0.4, x1_max - 0.4, 7)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for x1_vertex in vertex_positions:
    x2_squared = 2 * k * (x1_grid - x1_vertex)
    mask = x2_squared >= 0
    plt.plot(x1_grid[mask], np.sqrt(x2_squared[mask]), color="royalblue", alpha=0.8)
    plt.plot(x1_grid[mask], -np.sqrt(x2_squared[mask]), color="royalblue", alpha=0.8)

plt.title("Фазовые траектории при μ = +1")
plt.xlabel("x1, рад")
plt.ylabel("x2, рад/с")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)

plt.subplot(1, 2, 2)
for x1_vertex in vertex_positions:
    x2_squared = 2 * k * (x1_vertex - x1_grid)
    mask = x2_squared >= 0
    plt.plot(x1_grid[mask], np.sqrt(x2_squared[mask]), color="crimson", alpha=0.8)
    plt.plot(x1_grid[mask], -np.sqrt(x2_squared[mask]), color="crimson", alpha=0.8)

plt.title("Фазовые траектории при μ = -1")
plt.xlabel("x1, рад")
plt.ylabel("x2, рад/с")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.show()

# ============================================================
# Оптимальная фазовая траектория для перехода в точку (0.5; 0.5)
# ============================================================
plt.figure(figsize=(8, 5))

plt.plot(x1_1, x2_1, label="1-й участок: μ = -1", linewidth=2.2)
plt.plot(x1_2, x2_2, label="2-й участок: μ = +1", linewidth=2.2)

# Конечная фазовая траектория при mu = +1.
# Нижняя ветвь проходит через точку переключения, верхняя -- через конечную точку.
x1_line_lower = np.linspace(x1_vertex_final, x1_sw, 600)
inside_lower = x2f**2 + 2 * k * (x1_line_lower - x1f)
x2_line_lower = -np.sqrt(np.maximum(0.0, inside_lower))
plt.plot(x1_line_lower, x2_line_lower, "--", color="black", label="конечная траектория, нижняя ветвь")

x1_line_upper = np.linspace(x1_vertex_final, x1f, 600)
inside_upper = x2f**2 + 2 * k * (x1_line_upper - x1f)
x2_line_upper = np.sqrt(np.maximum(0.0, inside_upper))
plt.plot(x1_line_upper, x2_line_upper, ":", color="black", label="конечная траектория, верхняя ветвь")

plt.scatter(x10, x20, label="начальная точка", zorder=5)
plt.scatter(x1_sw, x2_sw, label="точка переключения", zorder=5)
plt.scatter(x1_vertex_final, 0, label="вершина конечной траектории", zorder=5)
plt.scatter(x1f, x2f, label="конечная точка (0.5; 0.5)", zorder=5)

plt.title("Оптимальная фазовая траектория")
plt.xlabel("x1, рад")
plt.ylabel("x2, рад/с")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# Численное моделирование методом Рунге-Кутты 4-го порядка
# ============================================================
def system_right_part(state, mu):
    x1, x2 = state
    return np.array([x2, k * mu])


def rk4_step(state, mu, h):
    k1 = system_right_part(state, mu)
    k2 = system_right_part(state + 0.5 * h * k1, mu)
    k3 = system_right_part(state + 0.5 * h * k2, mu)
    k4 = system_right_part(state + h * k3, mu)
    return state + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6


def simulate_segment(state0, t0, duration, mu, h):
    t_values = [t0]
    x1_values = [state0[0]]
    x2_values = [state0[1]]
    mu_values = [mu]

    state = state0.copy()
    t = t0
    t_end = t0 + duration

    while t < t_end - 1e-12:
        h_step = min(h, t_end - t)
        state = rk4_step(state, mu, h_step)
        t += h_step

        t_values.append(t)
        x1_values.append(state[0])
        x2_values.append(state[1])
        mu_values.append(mu)

    return np.array(t_values), np.array(x1_values), np.array(x2_values), np.array(mu_values), state


def simulate(h=0.001):
    state0 = np.array([x10, x20], dtype=float)

    t_a, x1_a, x2_a, mu_a, state_sw_num = simulate_segment(state0, 0.0, dt1, mu1, h)
    t_b, x1_b, x2_b, mu_b, state_final_num = simulate_segment(state_sw_num, dt1, dt2, mu2, h)

    t_values = np.concatenate([t_a, t_b[1:]])
    x1_values = np.concatenate([x1_a, x1_b[1:]])
    x2_values = np.concatenate([x2_a, x2_b[1:]])
    mu_values = np.concatenate([mu_a, mu_b[1:]])

    return t_values, x1_values, x2_values, mu_values, state_sw_num, state_final_num


t_num, x1_num, x2_num, mu_num, state_sw_num, state_final_num = simulate(h=0.001)

print()
print("Численное моделирование")
print("-----------------------")
print(f"x1_sw_num = {state_sw_num[0]:.9f} рад")
print(f"x2_sw_num = {state_sw_num[1]:.9f} рад/с")
print(f"t_sw_num = {dt1:.6f} с")
print(f"x1_final_num = {state_final_num[0]:.9f} рад")
print(f"x2_final_num = {state_final_num[1]:.9f} рад/с")
print(f"t_final_num = {t_total:.6f} с")

# Переходные процессы x1(t), x2(t)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(t_num, x1_num, label="x1(t)")
plt.axvline(dt1, linestyle="--", label=f"переключение: {dt1:.2f} с")
plt.axhline(x1f, linestyle=":", label="x1f = 0.5 рад")
plt.title("Переходный процесс по x1")
plt.xlabel("t, с")
plt.ylabel("x1, рад")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t_num, x2_num, label="x2(t)")
plt.axvline(dt1, linestyle="--", label=f"переключение: {dt1:.2f} с")
plt.axhline(x2f, linestyle=":", label="x2f = 0.5 рад/с")
plt.title("Переходный процесс по x2")
plt.xlabel("t, с")
plt.ylabel("x2, рад/с")
plt.legend()

plt.tight_layout()
plt.show()

# Закон управления mu(t)
plt.figure(figsize=(8, 3))
plt.step(t_num, mu_num, where="post")
plt.axvline(dt1, linestyle="--", label="момент переключения")
plt.title("Оптимальное управление μ(t)")
plt.xlabel("t, с")
plt.ylabel("μ")
plt.yticks([-1, 0, 1])
plt.legend()
plt.tight_layout()
plt.show()

print()
print("Сравнение аналитических и численных значений")
print("--------------------------------------------")
print(f"x1_sw: аналитически {x1_sw:.9f}, численно {state_sw_num[0]:.9f}")
print(f"x2_sw: аналитически {x2_sw:.9f}, численно {state_sw_num[1]:.9f}")
print(f"x1_final: требуется {x1f:.9f}, численно {state_final_num[0]:.9f}")
print(f"x2_final: требуется {x2f:.9f}, численно {state_final_num[1]:.9f}")
print(f"dt1 = {dt1:.6f} с")
print(f"dt2 = {dt2:.6f} с")
print(f"t_total = {t_total:.6f} с")
